Generate TES Geometry with CADQuery.

In [ ]:
import cadquery as cq
from jupyter_cadquery import *
from jupyter_cadquery.replay import replay, enable_replay
import math
import numpy as np

enable_replay(show_bbox=True, warning=False)
show_object = replay

# ASSUME UNITS ARE M

# beginning cylinder, scaling original design up by 2m, as inspired by Utili 2023
R = 2
tank_depth = 1.5
wall_thickness = 5e-3 # 5mm

# initial cylinder 
cyl = cq.Workplane("front").circle(R).extrude(tank_depth)

# split in half
cut_xz_plane = (
    cq.Workplane("XZ")
    .rect(5, 5)
    .extrude(wall_thickness/2, both=True) # extrudes by half the thickness in both Y directions
)
mixing_tank, bottom = cyl.split(cut_xz_plane, keepmixing_tank=True, keepBottom=True).solids(">Y"), cyl.split(cut_xz_plane, keepmixing_tank=False, keepBottom=True).solids("<<Y")
bottom = bottom.translate((0,-wall_thickness*4,0))

# split bottom in half 
cut_yz_plane = (
    cq.Workplane("YZ")
    .rect(5, 5)
    .extrude(wall_thickness/2, both=True) # extrudes by half the thickness in both X directions
)
inlet_tank, outlet_tank = bottom.split(cut_yz_plane).solids(">X"), bottom.split(cut_yz_plane).solids("<<X")
inlet_tank = inlet_tank.translate((wall_thickness*2,0,0))
outlet_tank = outlet_tank.translate((-wall_thickness*2,0,0))

# add fluid inlet and outlet 
fluid_radius = 0.35
fluid_depth = 0.55

inlet_tank_center = inlet_tank.faces(">>Z").val().Center()
inlet = cq.Workplane("XY").workplane(offset=tank_depth).center(inlet_tank_center.x, inlet_tank_center.y).circle(fluid_radius).extrude(fluid_depth)
inlet = inlet.translate((-R*0.15,-R*0.25,0))

outlet_tank_center = outlet_tank.faces(">>Z").val().Center()
outlet = cq.Workplane("XY").workplane(offset=tank_depth).center(outlet_tank_center.x, outlet_tank_center.y).circle(fluid_radius).extrude(fluid_depth)
outlet = outlet.translate((R*0.15,-R*0.25,0))

# PIPING 
tube_radius = 3e-1 / 2 # 9mm/2, inspired by Utili 2023
angle = 180 # angle of bend in degrees

def make_pipes(straight_length, bend_radius_growth_factor, x_growth_factor, y_growth_factor, path_type):
    """Function to make breeder pipes going from inlet to mixing tank, and from 
    mixing tank to outlet."""

    bend_radius = 0.4 + bend_radius_growth_factor

    # bend points
    ap1 = (bend_radius - math.cos(math.radians(angle/2))*bend_radius, straight_length + math.sin(math.radians(angle/2))*bend_radius)
    ap2 = (bend_radius - math.cos(math.radians(angle))*bend_radius, straight_length + math.sin(math.radians(angle))*bend_radius)

    # path selection 
    path = cq.Workplane("XY").vLine(straight_length).threePointArc(ap1, ap2).polarLine(straight_length, 90 - angle).consolidateWires()

    # rotate pipes
    pipe = cq.Workplane("XZ").circle(tube_radius).sweep(path)
    pipe = pipe.rotate((1, 0, 0), (0, 0, 0), 90)
    pipe = pipe.rotate((0, 0, 0), (0, 0, 1), 90)

    # offset pipes 
    x_translation_factor = 0.2 + x_growth_factor
    y_translation_factor = 0.2 + y_growth_factor

    if path_type == "inlet_to_mix":
        path_to_x_coordinate = 1
    else:
        path_to_x_coordinate = -1

    pipe = pipe.translate((path_to_x_coordinate*R*x_translation_factor,-R*y_translation_factor,0))

    return pipe


straight_lengths = np.linspace(5,7.5,4) # m
bend_radius_growth_factor = 0 # radius of bend
x_growth_factor = 0
y_growth_factor = 0
index = 0


pipes = []

# first layer pipes (inner most)
for straight_length in straight_lengths:
    index += 1

    inlet_pipe = make_pipes(straight_length, bend_radius_growth_factor,x_growth_factor, y_growth_factor, path_type="inlet_to_mix")
    outlet_pipe = make_pipes(straight_length, bend_radius_growth_factor,x_growth_factor, y_growth_factor, path_type="mix_to_outlet")
    pipes.append(inlet_pipe)
    pipes.append(outlet_pipe)

    if index == len(straight_lengths):
        y_growth_factor = 0
        bend_radius_growth_factor = 0
        index = 0
    else:
        y_growth_factor += 0.2
        bend_radius_growth_factor += 0.4

# second layer pipes
x_growth_factor += 0.3
for straight_length in straight_lengths[:3]:
    index += 1

    inlet_pipe = make_pipes(straight_length, bend_radius_growth_factor,x_growth_factor, y_growth_factor, path_type="inlet_to_mix")
    outlet_pipe = make_pipes(straight_length, bend_radius_growth_factor,x_growth_factor, y_growth_factor, path_type="mix_to_outlet")
    pipes.append(inlet_pipe)
    pipes.append(outlet_pipe)

    if index == len(straight_lengths[:3]):
        y_growth_factor = 0
        bend_radius_growth_factor = 0
        index = 0 
    else:
        y_growth_factor += 0.2
        bend_radius_growth_factor += 0.4


# third layer pipes
x_growth_factor += 0.3
for straight_length in straight_lengths[:2]:
    index += 1

    inlet_pipe = make_pipes(straight_length, bend_radius_growth_factor,x_growth_factor, y_growth_factor, path_type="inlet_to_mix")
    outlet_pipe = make_pipes(straight_length, bend_radius_growth_factor,x_growth_factor, y_growth_factor, path_type="mix_to_outlet")
    pipes.append(inlet_pipe)
    pipes.append(outlet_pipe)

    if index == len(straight_lengths[:2]):
        y_growth_factor = 0
        bend_radius_growth_factor = 0
        index = 0 
    else:
        y_growth_factor += 0.2
        bend_radius_growth_factor += 0.4

# merge into one volume
tes = mixing_tank.union(inlet_tank).union(outlet_tank).union(inlet).union(outlet)
for pipe in pipes:
    tes = tes.union(pipe)

tes_to_save = cq.Assembly()
tes_to_save.add(tes, name="fluid", color=cq.Color("green", alpha=0.3))

tes_to_save.save("tes_openfoam.step")
# tes_to_save.toCompound().exportBrep("tes_openfoam.brep")



Enabling jupyter_cadquery replay


True

In [ ]:
import cadquery as cq
from jupyter_cadquery import *
from jupyter_cadquery.replay import replay, enable_replay
import math
import numpy as np

enable_replay(show_bbox=True, warning=False)
show_object = replay

# ASSUME UNITS ARE M

# beginning cylinder, scaling original design up by 2m, as inspired by Utili 2023
R = 2
tank_depth = 1.5
wall_thickness = 5e-3 # 5mm

# initial cylinder 
cyl = cq.Workplane("front").circle(R).extrude(tank_depth)

# split in half
cut_xz_plane = (
    cq.Workplane("XZ")
    .rect(5, 5)
    .extrude(wall_thickness/2, both=True) # extrudes by half the thickness in both Y directions
)
mixing_tank, bottom = cyl.split(cut_xz_plane, keepmixing_tank=True, keepBottom=True).solids(">Y"), cyl.split(cut_xz_plane, keepmixing_tank=False, keepBottom=True).solids("<<Y")
bottom = bottom.translate((0,-wall_thickness*4,0))

# split bottom in half 
cut_yz_plane = (
    cq.Workplane("YZ")
    .rect(5, 5)
    .extrude(wall_thickness/2, both=True) # extrudes by half the thickness in both X directions
)
inlet_tank, outlet_tank = bottom.split(cut_yz_plane).solids(">X"), bottom.split(cut_yz_plane).solids("<<X")
inlet_tank = inlet_tank.translate((wall_thickness*2,0,0))
outlet_tank = outlet_tank.translate((-wall_thickness*2,0,0))

# add fluid inlet and outlet 
fluid_radius = 0.35
fluid_depth = 0.55

inlet_tank_center = inlet_tank.faces(">>Z").val().Center()
inlet = cq.Workplane("XY").workplane(offset=tank_depth).center(inlet_tank_center.x, inlet_tank_center.y).circle(fluid_radius).extrude(fluid_depth)
inlet = inlet.translate((-R*0.15,-R*0.25,0))

outlet_tank_center = outlet_tank.faces(">>Z").val().Center()
outlet = cq.Workplane("XY").workplane(offset=tank_depth).center(outlet_tank_center.x, outlet_tank_center.y).circle(fluid_radius).extrude(fluid_depth)
outlet = outlet.translate((R*0.15,-R*0.25,0))

# PIPING 
tube_radius = 3e-1 / 2 # 9mm/2, inspired by Utili 2023
angle = 180 # angle of bend in degrees

def make_pipes(straight_length, bend_radius_growth_factor, x_growth_factor, y_growth_factor, path_type):
    """Function to make breeder pipes going from inlet to mixing tank, and from 
    mixing tank to outlet."""

    bend_radius = 0.4 + bend_radius_growth_factor

    # bend points
    ap1 = (bend_radius - math.cos(math.radians(angle/2))*bend_radius, straight_length + math.sin(math.radians(angle/2))*bend_radius)
    ap2 = (bend_radius - math.cos(math.radians(angle))*bend_radius, straight_length + math.sin(math.radians(angle))*bend_radius)

    # path selection 
    path = cq.Workplane("XY").vLine(straight_length).threePointArc(ap1, ap2).polarLine(straight_length, 90 - angle).consolidateWires()

    # rotate pipes
    pipe = cq.Workplane("XZ").circle(tube_radius).sweep(path, isFrenet=True)

    pipe = pipe.rotate((1, 0, 0), (0, 0, 0), 90)
    pipe = pipe.rotate((0, 0, 0), (0, 0, 1), 90)

    # offset pipes 
    x_translation_factor = 0.2 + x_growth_factor
    y_translation_factor = 0.2 + y_growth_factor

    if path_type == "inlet_to_mix":
        path_to_x_coordinate = 1
    else:
        path_to_x_coordinate = -1

    pipe = pipe.translate((path_to_x_coordinate*R*x_translation_factor,-R*y_translation_factor,0))

    return pipe

straight_lengths = np.linspace(5,7.5,4) # m
bend_radius_growth_factor = 0 # radius of bend
x_growth_factor = 0
y_growth_factor = 0
index = 0

pipes = []

# first layer pipes (inner most)
for straight_length in straight_lengths:
    index += 1

    inlet_pipe = make_pipes(straight_length, bend_radius_growth_factor,x_growth_factor, y_growth_factor, path_type="inlet_to_mix")
    outlet_pipe = make_pipes(straight_length, bend_radius_growth_factor,x_growth_factor, y_growth_factor, path_type="mix_to_outlet")
    pipes.append(inlet_pipe)
    pipes.append(outlet_pipe)

    if index == len(straight_lengths):
        y_growth_factor = 0
        bend_radius_growth_factor = 0
        index = 0
    else:
        y_growth_factor += 0.2
        bend_radius_growth_factor += 0.4

# second layer pipes
x_growth_factor += 0.3
for straight_length in straight_lengths[:3]:
    index += 1

    inlet_pipe = make_pipes(straight_length, bend_radius_growth_factor,x_growth_factor, y_growth_factor, path_type="inlet_to_mix")
    outlet_pipe = make_pipes(straight_length, bend_radius_growth_factor,x_growth_factor, y_growth_factor, path_type="mix_to_outlet")
    pipes.append(inlet_pipe)
    pipes.append(outlet_pipe)

    if index == len(straight_lengths[:3]):
        y_growth_factor = 0
        bend_radius_growth_factor = 0
        index = 0 
    else:
        y_growth_factor += 0.2
        bend_radius_growth_factor += 0.4


# third layer pipes
x_growth_factor += 0.3
for straight_length in straight_lengths[:2]:
    index += 1

    inlet_pipe = make_pipes(straight_length, bend_radius_growth_factor,x_growth_factor, y_growth_factor, path_type="inlet_to_mix")
    outlet_pipe = make_pipes(straight_length, bend_radius_growth_factor,x_growth_factor, y_growth_factor, path_type="mix_to_outlet")
    pipes.append(inlet_pipe)
    pipes.append(outlet_pipe)

    if index == len(straight_lengths[:2]):
        y_growth_factor = 0
        bend_radius_growth_factor = 0
        index = 0 
    else:
        y_growth_factor += 0.2
        bend_radius_growth_factor += 0.4

# # SHELLING
# inlets = inlet_tank.union(inlet)
# outlets = outlet_tank.union(outlet)

# inlet_shell = inlets.shell(wall_thickness)
# outlet_shell = outlets.shell(wall_thickness)
# mixing_shell = mixing_tank.shell(wall_thickness)

# for pipe in pipes: 
#     new_pipe = pipe.translate((0,0,wall_thickness))
#     inlet_shell = inlet_shell.cut(new_pipe)
#     outlet_shell = outlet_shell.cut(new_pipe)
#     mixing_shell = mixing_shell.cut(new_pipe)

# merge into one volume
# insulating_walls = inlet_shell.union(outlet_shell).union(mixing_shell)
fluid = mixing_tank.union(inlet_tank).union(outlet_tank).union(inlet).union(outlet)
for pipe in pipes: 
    fluid = fluid.union(pipe)

# shell pipes but subtract wall thickness at its juncture with mixing tank and inlets/outlets 
# cut_xy_plane = (
#     cq.Workplane("XY")
#     .rect(5, 5)
#     .extrude(-wall_thickness) # extrudes by the thickness in -Z direction
# )

pipe_counter = 0
pipe_shells = pipes[0].solids("<Z").faces(">Z").shell(wall_thickness)
for pipe in pipes:
    if pipe_counter != 0:
        pipe_shells = pipe_shells.union(pipe.solids("<Z").faces(">Z").shell(wall_thickness)) # must shell BEFORE union 
    pipe_counter += 1 

tes = cq.Assembly()
tes.add(fluid, name="fluid", color=cq.Color("green", alpha=0.3))
tes.add(pipe_shells, name="membrane", color=cq.Color("red", alpha=0.3))
# tes.add(insulating_walls, name="walls", color=cq.Color("blue", alpha=0.3))

tes.toCompound().exportBrep("tes_festim.brep")


Enabling jupyter_cadquery replay


True